# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

FILE_NAME = "loo_summary_merfish"
CSV_PATH = f"../results/{FILE_NAME}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,200,0.068864,0.230110,0.115,0.217391,0.025,0.019022,14.429766,14.971449,521.428223
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,200,0.460826,0.497896,0.165,0.696970,0.115,0.293658,23.868064,27.635377,1234.587036
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,200,0.399375,0.309090,0.210,0.857143,0.180,0.069700,17.152134,18.944778,1589.633545
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,200,0.701164,0.642562,0.280,0.928571,0.260,0.000000,36.351183,38.485640,2399.222656
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,200,0.265527,0.311898,0.185,0.945946,0.175,0.003236,17.114140,17.957415,1034.296997


In [8]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "direction_match_k": +1,
    "edistance_local": -1,
    #"rmse": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    #"mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    #"cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    #"MintFlow",
    "cellina (ablated)",
    #"cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'CPA', 'scGen'],
      dtype=object)

In [9]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
per_slide = (
    df.groupby(["model_name", "sid"])[list(METRICS)]
    .mean()
    .reset_index()
)

agg = (
    per_slide.groupby("model_name")[list(METRICS)]
    .agg(["mean", "std"])
    .reindex(MODEL_ORDER)
)
agg

spearman             pearson           precision            \
                       mean       std      mean       std      mean       std   
model_name                                                                      
mean shift         0.397703  0.019700  0.361712  0.021069  0.249167  0.025007   
CPA                0.759975  0.019218  0.757909  0.022975  0.418833  0.035058   
scGen              0.704745  0.026939  0.721444  0.013808  0.384167  0.029366   
cellina (ablated)  0.670783  0.047252  0.672862  0.039284  0.316000  0.032600   
cellina            0.801184  0.032335  0.791315  0.025502  0.487333  0.020213   

                  direction_match_k           edistance_local            
                               mean       std            mean       std  
model_name                                                               
mean shift                 0.194167  0.017765       27.562882  0.602780  
CPA                        0.399333  0.035030        8.194920  0.601247  
scGen                      0.373667  0.031086        6.686902  0.752906  
cellina (ablated)          0.290333  0.034038        9.051087  0.774720  
cellina                    0.471167  0.022969        8.346333  0.720825

In [10]:
def find_best(agg: pd.DataFrame) -> dict:
    """Return {metric: model_name_of_best} honouring direction."""
    best = {}
    for metric, direction in METRICS.items():
        means = agg[(metric, "mean")].dropna()
        if means.empty:
            continue
        best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s


best = find_best(agg)

table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])
for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        table.loc[model, col] = format_cell(mean, std, bold=(best.get(metric) == model))

table.index.name = "Method"
table

,Spearman $\uparrow$,Pearson $\uparrow$,Precision $\uparrow$,Direction Match (K) $\uparrow$,E-dist (local) $\downarrow$
Method,,,,,
mean shift,0.398 $\pm$ 0.020,0.362 $\pm$ 0.021,0.249 $\pm$ 0.025,0.194 $\pm$ 0.018,27.563 $\pm$ 0.603
CPA,0.760 $\pm$ 0.019,0.758 $\pm$ 0.023,0.419 $\pm$ 0.035,0.399 $\pm$ 0.035,8.195 $\pm$ 0.601
scGen,0.705 $\pm$ 0.027,0.721 $\pm$ 0.014,0.384 $\pm$ 0.029,0.374 $\pm$ 0.031,\textbf{6.687 $\pm$ 0.753}
cellina (ablated),0.671 $\pm$ 0.047,0.673 $\pm$ 0.039,0.316 $\pm$ 0.033,0.290 $\pm$ 0.034,9.051 $\pm$ 0.775
cellina,\textbf{0.801 $\pm$ 0.032},\textbf{0.791 $\pm$ 0.026},\textbf{0.487 $\pm$ 0.020},\textbf{0.471 $\pm$ 0.023},8.346 $\pm$ 0.721


In [11]:
# Render as LaTeX (booktabs). escape=False keeps our \textbf and math arrows.
latex = table.to_latex(
    escape=False,
    column_format="l" + "c" * table.shape[1],
    caption=(
        "Leave-one-celltype-out performance (top 200 DEGs). For each slide we "
        "first average over the held-out cell types, then report mean $\\pm$ std "
        f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
    ),
    label=f"tab:{FILE_NAME}",
)
latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
print(latex)

(OUT_DIR / f"{FILE_NAME}_table.tex").write_text(latex)

\begin{table}[t]
\centering
\caption{Leave-one-celltype-out performance (top 200 DEGs). For each slide we first average over the held-out cell types, then report mean $\pm$ std across 3 slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_merfish}
\begin{tabular}{lccccc}
\toprule
 & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & Direction Match (K) $\uparrow$ & E-dist (local) $\downarrow$ \\
Method &  &  &  &  &  \\
\midrule
mean shift & 0.398 $\pm$ 0.020 & 0.362 $\pm$ 0.021 & 0.249 $\pm$ 0.025 & 0.194 $\pm$ 0.018 & 27.563 $\pm$ 0.603 \\
CPA & 0.760 $\pm$ 0.019 & 0.758 $\pm$ 0.023 & 0.419 $\pm$ 0.035 & 0.399 $\pm$ 0.035 & 8.195 $\pm$ 0.601 \\
scGen & 0.705 $\pm$ 0.027 & 0.721 $\pm$ 0.014 & 0.384 $\pm$ 0.029 & 0.374 $\pm$ 0.031 & \textbf{6.687 $\pm$ 0.753} \\
cellina (ablated) & 0.671 $\pm$ 0.047 & 0.673 $\pm$ 0.039 & 0.316 $\pm$ 0.033 & 0.290 $\pm$ 0.034 & 9.051 $\pm$ 0.775 \\
cellina & \textbf{0.801 $\pm$ 0.032} & \textbf{0.791 $\pm$ 0.026} & \textbf{0.

1107

In [12]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method            | Spearman ↑        | Pearson ↑         | Precision ↑       | Direction Match (K) ↑   | E-dist (local) ↓   |
|:------------------|:------------------|:------------------|:------------------|:------------------------|:-------------------|
| mean shift        | 0.398 ± 0.020     | 0.362 ± 0.021     | 0.249 ± 0.025     | 0.194 ± 0.018           | 27.563 ± 0.603     |
| CPA               | 0.760 ± 0.019     | 0.758 ± 0.023     | 0.419 ± 0.035     | 0.399 ± 0.035           | 8.195 ± 0.601      |
| scGen             | 0.705 ± 0.027     | 0.721 ± 0.014     | 0.384 ± 0.029     | 0.374 ± 0.031           | **6.687 ± 0.753**  |
| cellina (ablated) | 0.671 ± 0.047     | 0.673 ± 0.039     | 0.316 ± 0.033     | 0.290 ± 0.034           | 9.051 ± 0.775      |
| cellina           | **0.801 ± 0.032** | **0.791 ± 0.026** | **0.487 ± 0.020** | **0.471 ± 0.023**       | 8.346 ± 0.721      |
